In [1]:
import os
import numpy as np
import pandas as pd
import dateutil.relativedelta
import datetime
from datetime import date, datetime
import warnings
import functools
import gc
import math
import sqldf
from glob import iglob # Global directory
from itertools import product
from typing import List, Tuple, Iterable
from tqdm import tqdm # Show data processing bar
import re
from datetime import datetime as dt
from dateutil.relativedelta import relativedelta

from IPython.core.interactiveshell import InteractiveShell
from IPython.display import display, HTML # Display image on jupyter notebook
InteractiveShell.ast_node_interactivity="all"
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)  # or 1000
pd.set_option('display.max_rows', None)  # or 1000
pd.set_option('display.max_colwidth', None)  # or 199
pd.set_option('display.max_columns', None)

root_directory = './Input/' 

## Load and process Mck-Oneoff Dataset

In [2]:
# Function to load and preprocess the MCK OneOff data
def load_and_preprocess_mck_data(file_path, date_column, date_format='mixed'):
    """Loads and preprocesses MCK OneOff data."""
    df = pd.read_csv(file_path)
    # Convert the date column to datetime
    df[date_column] = pd.to_datetime(df[date_column], format=date_format)
    # Extract Year-Month from the date
    df['Year_Month_Obs'] = df[date_column].dt.strftime('%Y-%m')
    # Reset index for clean processing
    return df.reset_index(drop=True)

# Function to filter data by date range
def filter_data_by_date_range(df, date_column, start_date, end_date):
    """Filters the DataFrame based on a date range."""
    df[date_column] = pd.to_datetime(df[date_column])  # Ensure column is datetime
    return df[(df[date_column] > start_date) & (df[date_column] <= end_date)].reset_index(drop=True)

# Function to select relevant columns for analysis
def select_columns(df, columns):
    """Selects a subset of columns for analysis."""
    return df[columns]

# Main workflow
def process_mck_oneoff_data(file_path):
    """Complete workflow for processing MCK OneOff data."""
    # Load and preprocess the data
    mck_data = load_and_preprocess_mck_data(file_path, date_column='DataDate')
    
    # Filter data by date range (August 2022 to August 2023)
    mck_data_filtered = filter_data_by_date_range(
        mck_data, date_column='Year_Month_Obs', start_date='2022-08', end_date='2023-08'
    )
    
    # Select relevant columns for analysis
    columns_of_interest = [
        'LoanNumber', 'LoanType', 'InterestRate', 'OriginalInterestRate',
        'FICO_Score', 'Original_Ltv_Calc', 'CurrentLTV','PrincipalBalance',
        'TermOfLoan', 'RemainingTerm', 'Year_Month_Obs', 'PropertyZipCode', 'PropertyState', 'MailingZip'
        
    ]
    mck_data_selected = select_columns(mck_data_filtered, columns_of_interest)
    mck_oneoff_extract_column_unique = mck_data_selected.sort_values(['Year_Month_Obs']).groupby(['LoanNumber']).last().reset_index()

    print(f"Filtered Data Shape: {mck_oneoff_extract_column_unique.shape}")
    
    return mck_oneoff_extract_column_unique



## Load and process Refinance data

In [3]:
import os
import pandas as pd

def load_and_clean_file(file_path, final_columns=None):
    """
    Load and clean a single Excel file by matching and renaming columns dynamically.

    Args:
    - file_path (str): Path to the Excel file.
    - final_columns (dict): Dictionary of target column names as keys and possible source names as values.

    Returns:
    - pd.DataFrame: Cleaned DataFrame with uniform column names.
    """
    df = pd.read_excel(file_path)

    # Create a mapping of existing columns to target columns
    column_mapping = {}
    for target_col, possible_names in final_columns.items():
        for col in possible_names:
            if col in df.columns:
                column_mapping[col] = target_col
                break

    # Rename columns
    df = df.rename(columns=column_mapping)

    # Filter to keep only the target columns
    return df[list(final_columns.keys())]

def process_all_files(directory, final_columns):
    """
    Process all files in a directory, dynamically handling columns.

    Args:
    - directory (str): Directory containing the Excel files.
    - final_columns (dict): Dictionary of target column names and possible source names.

    Returns:
    - pd.DataFrame: Combined DataFrame of all files.
    """
    all_data = []

    # Iterate through all files in the directory
    for file_name in os.listdir(directory):
        file_path = os.path.join(directory, file_name)

        # Process only Excel files
        if file_name.endswith(('.xlsx', '.xls')):
            try:
                # Process the file
                df = load_and_clean_file(file_path, final_columns=final_columns)
                all_data.append(df)
            except Exception as e:
                print(f"Error processing {file_name}: {e}")

    # Combine all data
    return pd.concat(all_data, axis=0, ignore_index=True) if all_data else pd.DataFrame()

def filter_and_clean_final_data(data):
    """
    Filter and clean the combined data as per the additional conditions.

    Args:
    - data (pd.DataFrame): Combined DataFrame.

    Returns:
    - pd.DataFrame: Cleaned DataFrame after filtering.
    """
    # Filter out rows where 'Reason_Closed' is 'REO' or 'SS'
    data = data[(data["Reason_Closed"] != "REO") & (data["Reason_Closed"] != "SS")]

    # Ensure 'Reason_Closed_Redifn' is string, lowercase, and stripped
    data['Reason_Closed_Redifn'] = data['Reason_Closed_Redifn'].astype(str)
    data['Reason_Closed_Redifn'] = data['Reason_Closed_Redifn'].str.lower()
    data['Reason_Closed_Redifn'] = data['Reason_Closed_Redifn'].str.strip()

    # Remove rows where 'Reason_Closed_Redifn' is 'nan'
    data = data[data['Reason_Closed_Redifn'] != 'nan']
    data = data[data["Reason_Closed_Redifn"].isin(["off market","not found","for rent",
                                                                                       "not available","offmarket","house for rent",
                                                                                       "rent"])]
   
    data.reset_index(inplace=True)
    data["Loan_Number"]=data["Loan_Number"].astype("int64")

    # Remove duplicates based on column 'LoanNumber' keeping the most recent Date_Paid_Off
    data = data.sort_values(by='Date_Paid_Off', ascending=False).drop_duplicates(subset='Loan_Number', keep='first')

    return data



In [4]:
%%time

file_path = os.path.join(root_directory, 'MCK_ONEOFF_11_02_2023.csv')
processed_mck_data = process_mck_oneoff_data(file_path)
print(processed_mck_data.head())
print(processed_mck_data.shape)


Filtered Data Shape: (164246, 14)
   LoanNumber      LoanType  InterestRate  OriginalInterestRate  FICO_Score  \
0         791  CONVENTIONAL         2.750                 2.750       777.0   
1        1235  CONVENTIONAL         2.875                 2.875       608.0   
2        1262  CONVENTIONAL         2.875                 2.875       758.0   
3        1378  CONVENTIONAL         4.875                 4.875       503.0   
4        1571  CONVENTIONAL         3.500                 3.500       801.0   

   Original_Ltv_Calc  CurrentLTV  PrincipalBalance  TermOfLoan  RemainingTerm  \
0              66.67       22.06          69477.71         180             55   
1              63.15       22.95          53245.69         180             57   
2              75.00       25.22         138728.50         180             58   
3              80.00       79.90         122249.50         360            359   
4              96.63       36.23          37677.01         180             57   

  Ye

In [5]:
# Define the root directory

payoff_reason_data_path = os.path.join(root_directory, 'Payoffs Reasons')

# Define target columns and possible source names
final_columns = {
    'Loan_Number': ['Loan Number', 'LoanNumber'],
    'Date_Paid_Off': ['Date Paid Off'],
    'Reason_Closed_Redifn': ['Reason Closed_Redifn'],
    'Reason_Closed': ['Reason Closed'],
    'Note_Rate': ['Note Rate'],
    'UPB_At_Payoff': ['UPB At Payoff', 'UPB At  Payoff'],
    'Original_Value': ['Original Value', 'Original  Value'],
    'Current_Value': ['Current Value'],
    'Current_LTV_%': ['Current LTV % ', 'Current LTV %'],
    'Original_LTV_%': ['Original LTV % ', 'Original LTV %'],
    'Term_of_Loan': ['Term of Loan'],
    'Remaining_Term': ['Remaining Term', 'Remaining  Term'],
    'Investor': ['Investor'],
    'Seller': ['Seller'],
    'Credit_Score': ['Credit Score']
}

# Process all files in the folder
all_payoff_reason_data = process_all_files(payoff_reason_data_path, final_columns)
# Filter and clean the combined data
all_payoff_reason_data = filter_and_clean_final_data(all_payoff_reason_data)


# Display the combined data
print(all_payoff_reason_data.shape)
print(all_payoff_reason_data.head())

(1937, 16)
      index  Loan_Number Date_Paid_Off Reason_Closed_Redifn Reason_Closed  \
1727   5173   1401487882    2023-08-31           off market           PIF   
1600   4719   1401157873    2023-08-31           off market           PIF   
1657   4911   7801500062    2023-08-31           off market           PIF   
1624   4800    960225917    2023-08-31           off market           PIF   
1623   4797    640235801    2023-08-31           off market           PIF   

      Note_Rate  UPB_At_Payoff  Original_Value  Current_Value  Current_LTV_%  \
1727       3.75      245768.78        254375.0       254375.0            0.0   
1600       4.50      148639.84        183218.0       197614.0            0.0   
1657       6.13      603448.98        612000.0       685000.0            0.0   
1624       3.88      105810.67        112500.0       167000.0            0.0   
1623       3.38      171958.76        183100.0       400000.0            0.0   

      Original_LTV_%  Term_of_Loan  Remaining

## Merging Payoff Reason Data & Mck One Off Data

In [6]:
def merge_and_process_data(mck_oneoff_data, payoff_reason_data):
    """
    Merges the MCK One-Off data with the Payoff Reasons data and processes date-related columns.

    Args:
    - mck_oneoff_data (pd.DataFrame): The unique MCK One-Off extract columns.
    - payoff_reason_data (pd.DataFrame): The cleaned Payoff Reasons DataFrame.

    Returns:
    - pd.DataFrame: Processed DataFrame with selected columns and computed date fields.
    """
    # Merge the two datasets on LoanNumber
    merged_data = mck_oneoff_data.merge(
        payoff_reason_data,
        how="left",
        left_on="LoanNumber",
        right_on="Loan_Number"
    )

    # Debugging shapes of the data
    print(f"MCK One-Off Extract Shape: {mck_oneoff_data.shape}")
    print(f"Payoff Reason Data Shape: {payoff_reason_data.shape}")

    # Convert 'Date_Paid_Off' to datetime and format to 'YYYY-MM'
    merged_data["Date_Paid_Off"] = pd.to_datetime(merged_data["Date_Paid_Off"], errors="coerce")
    merged_data["Date_Paid_Off"] = merged_data["Date_Paid_Off"].dt.strftime("%Y-%m")

    # Create 'Event_Date': fill missing 'Date_Paid_Off' with default and format
    merged_data["Event_Date"] = merged_data["Date_Paid_Off"].fillna("2023-08")
    merged_data["Event_Date"] = pd.to_datetime(merged_data["Event_Date"], format="%Y-%m")
    merged_data["Event_Date"] = merged_data["Event_Date"].dt.strftime("%Y-%m")

    # Calculate 'Interest_Rate_Date' (two months prior to 'Event_Date')
    merged_data["Interest_Rate_Date"] = pd.to_datetime(merged_data["Event_Date"], format="%Y-%m") - pd.DateOffset(months=2)
    merged_data["Interest_Rate_Date"] = merged_data["Interest_Rate_Date"].dt.strftime("%Y-%m")

    # Select final columns for output
    final_columns = [
        'LoanNumber', 'LoanType', 'InterestRate', 'OriginalInterestRate', 'FICO_Score',
        'Original_Ltv_Calc', 'CurrentLTV', 'TermOfLoan', 'RemainingTerm', 'Interest_Rate_Date',
        'Year_Month_Obs', 'PropertyZipCode', 'PropertyState', 'MailingZip', 'PrincipalBalance'
    ]

       
    processed_data = merged_data[final_columns]

    print(f"Merged Data Shape: {processed_data.shape}")

    return processed_data

# Usage Example:
mck_oneoff_specific_column = merge_and_process_data(processed_mck_data, all_payoff_reason_data)

# Display the first row of the processed data
print(mck_oneoff_specific_column.head(1))
print(mck_oneoff_specific_column.LoanNumber.nunique())


MCK One-Off Extract Shape: (164246, 14)
Payoff Reason Data Shape: (1937, 16)
Merged Data Shape: (164246, 15)
   LoanNumber      LoanType  InterestRate  OriginalInterestRate  FICO_Score  \
0         791  CONVENTIONAL          2.75                  2.75       777.0   

   Original_Ltv_Calc  CurrentLTV  TermOfLoan  RemainingTerm  \
0              66.67       22.06         180             55   

  Interest_Rate_Date Year_Month_Obs PropertyZipCode PropertyState MailingZip  \
0            2023-06     2023-08-01            8260            NJ      08081   

   PrincipalBalance  
0          69477.71  
164246


## Getting interest rates from Curinos

In [7]:
# Function to bucket and transform columns
def bucket_and_transform_data(data):
    """
    Apply bucketing for FICO scores, Current LTV, and compute additional fields like Year, Month, and Term.
    """
    # Convert Interest_Rate_Date to Year and Month
    data["Interest_Rate_Date"] = pd.to_datetime(data["Interest_Rate_Date"], format='%Y-%m')
    data['Year'] = data['Interest_Rate_Date'].dt.year
    data['Month'] = data['Interest_Rate_Date'].dt.month
    
    # Determine Term (30 or 15 years)
    data['Term'] = np.where((data['RemainingTerm'] / 12) > 18, 30, 15)
    
    # Bucketing FICO Scores
    fico_conditions = [
        data['FICO_Score'].between(0, 620),
        data['FICO_Score'].between(620, 640),
        data['FICO_Score'].between(640, 660),
        data['FICO_Score'].between(660, 680),
        data['FICO_Score'].between(680, 700),
        data['FICO_Score'].between(700, 720),
        data['FICO_Score'].between(720, 740),
        data['FICO_Score'].between(740, 760),
        data['FICO_Score'].between(760, 780),
        data['FICO_Score'].ge(780),
    ]
    fico_choices = ["<620", ">=620", ">=640", ">=660", ">=680", ">=700", ">=720", ">=740", ">=760", ">=780"]
    data["FICO"] = np.select(fico_conditions, fico_choices, default=np.nan)
    
    # Bucketing Current LTV
    ltv_conditions = [
        data['CurrentLTV'].between(0, 30),
        data['CurrentLTV'].between(30, 60),
        data['CurrentLTV'].between(60, 70),
        data['CurrentLTV'].between(70, 75),
        data['CurrentLTV'].between(75, 80),
        data['CurrentLTV'].between(80, 85),
        data['CurrentLTV'].between(85, 90),
        data['CurrentLTV'].between(90, 95),
        data['CurrentLTV'].ge(95),
    ]
    ltv_choices = ["<=30", "<=60", "<=70", "<=75", "<=80", "<=85", "<=90", "<=95", ">95"]
    data["LTV"] = np.select(ltv_conditions, ltv_choices, default=np.nan)

    return data

# Apply transformations to MCK dataset
mck_oneoff_specific_column = bucket_and_transform_data(mck_oneoff_specific_column)

# Select final columns for merging
mck_oneoff_final_df = mck_oneoff_specific_column[['LoanNumber', 'LoanType', 'Year', 'Month', 'Term', 'FICO', 'LTV']]
mck_oneoff_final_df[["Year", "Month", "Term"]] = mck_oneoff_final_df[["Year", "Month", "Term"]].apply(pd.to_numeric)

# Load and preprocess interest rate matrix
interest_rate_matrix = pd.read_csv("./Input/Curinos_Interest_Rate_File_2022-2024.csv")
interest_rate_matrix['Term'] = np.where(interest_rate_matrix['Term'] == '15 Year', 15, 30)
interest_rate_matrix.rename(columns={'Product': 'LoanType', 'ficorange': 'FICO', 'ltvrange': 'LTV', 'avg(rate)': 'Avg_Rate'}, inplace=True)
interest_rate_matrix['LoanType'] = interest_rate_matrix['LoanType'].str.upper()
interest_rate_matrix = interest_rate_matrix[['LoanType', 'Year', 'Month', 'Term', 'FICO', 'LTV', 'Avg_Rate']]


# Merge datasets on keys
merged_data = mck_oneoff_final_df.merge(
    interest_rate_matrix, 
    how='left', 
    on=['LoanType', 'Year', 'Month', 'Term', 'FICO', 'LTV']
)

# Drop duplicates
merged_data.drop_duplicates(inplace=True)

# Fill missing Avg_Rate values with group means
group_means = interest_rate_matrix.groupby(['LoanType', 'Year', 'Month', 'Term', 'FICO'])['Avg_Rate'].mean().reset_index()
merged_data = merged_data.merge(group_means, how='left', on=['LoanType', 'Year', 'Month', 'Term', 'FICO'], suffixes=('', '_Mean'))

# Fill missing interest rates
merged_data['Orginal_Interest_Rate'] = merged_data['Avg_Rate'].fillna(merged_data['Avg_Rate_Mean'])

impute_values = dict(merged_data.groupby("LoanType")["Orginal_Interest_Rate"].mean())

# Replace NaN values
impute_values = {
    key: (np.round(impute_values['CONVENTIONAL'],5) if isinstance(value, float) and math.isnan(value) else np.round(value,5))
    for key, value in impute_values.items()
}

# Special handling for specific Loan Types
merged_data['Orginal_Interest_Rate'] = np.where(
    merged_data['LoanType'] == "PMI", 
    impute_values['PMI'], 
    merged_data['Orginal_Interest_Rate']
)
merged_data['Orginal_Interest_Rate'] = np.where(
    merged_data['LoanType'] == "CONVENTIONAL", 
    impute_values['CONVENTIONAL'], 
    merged_data['Orginal_Interest_Rate']
)

# Final dataset
final_interest_rate = merged_data[['LoanNumber', 'LoanType', "Year", "Month", "Term", "FICO", "LTV", "Orginal_Interest_Rate"]]
final_interest_rate.head(1)



,LoanNumber,LoanType,Year,Month,Term,FICO,LTV,Orginal_Interest_Rate
0,791,CONVENTIONAL,2023,6,15,>=760,<=30,6.61153


In [8]:
def transform_credit_report_to_time_series(BSI_CreditReporting, Year_Month_Obs=None):
    print("After implementing CB_RPTD_STA, CB_RPTG_STA and CLOSE_CD_DB000 check")
    filtered_df = BSI_CreditReporting.copy()
    print(filtered_df.shape)
    if not Year_Month_Obs:
        Year_Month_Obs=[
            '2023-01',
            '2023-02',
            '2023-03',
            '2023-04',
            '2023-05',
            '2023-06',
            '2023-07',
            '2023-08',
            '2023-09',
            '2023-10',
            '2023-11',
            '2023-12',
            '2022-01',
            '2022-02',
            '2022-03',
            '2022-04',
            '2022-05',
            '2022-06',
            '2022-07',
            '2022-08',
            '2022-09',
            '2022-10',
            '2022-11',
            '2022-12',
            '2021-01',
            '2021-02',
            '2021-03',
            '2021-04',
            '2021-05',
            '2021-06',
            '2021-07',
            '2021-08',
            '2021-09',
            '2021-10',
            '2021-11',
            '2021-12',
        ]
 
    All_LoanNumber=[]
    All_Year_Month_Obs=[]
    All_History=[]
    for loan in filtered_df['ACCT_NBR']:
        History=filtered_df[filtered_df['ACCT_NBR']==loan].iloc[:,-37:-1]
        LoanNumber=[loan]*36
        x1=LoanNumber
        x2=Year_Month_Obs
        x3=History.iloc[0,:].to_list()
        All_LoanNumber.extend(x1)
        All_Year_Month_Obs.extend(x2)
        All_History.extend(x3)
 
    data = {'LoanNumber': All_LoanNumber,
            'Year_Month_Obs': All_Year_Month_Obs,
            'CREDIT_BUREAU_PAYMENT_HISTORY':All_History}
    BSI_CreditReporting = pd.DataFrame()
    BSI_CreditReporting = pd.DataFrame(data)
    print('>>>>> Credit Report Transformed to Timeseries <<<<<<')
    return BSI_CreditReporting
 

def process_credit_report(file_path, transform_func):
    """
    Process the BSI Credit Reporting data into a summarized format.

    Args:
        file_path (str): Path to the Excel file containing the credit report data.
        transform_func (function): Function to transform credit report data into a time series.

    Returns:
        pd.DataFrame: A dataframe summarizing the credit bureau payment history.
    """
    # Load data
    bsi_credit_report = pd.read_excel(file_path)
    
    # Filter rows with 'CBR_LAST_REPORT_DATE' > '2022-01'
    yes_reportclosedate = bsi_credit_report[bsi_credit_report['CBR_LAST_REPORT_DATE'] > '2022-01'].reset_index()

    # Add start_date and end_date columns
    yes_reportclosedate['start_date'] = yes_reportclosedate.CBR_LAST_REPORT_DATE.apply(
        lambda x: dt.strptime(dt.strftime(x + relativedelta(months=-13), '%Y-%m'), '%Y-%m')
    )
    yes_reportclosedate['end_date'] = yes_reportclosedate.CBR_LAST_REPORT_DATE.apply(
        lambda x: dt.strptime(dt.strftime(x + relativedelta(months=-2), '%Y-%m'), '%Y-%m')
    )
    
    # Transform data to time series format
    BSI_CreditReporting_yesclosedate = transform_func(yes_reportclosedate)

    # Merge additional columns
    BSI_CreditReporting_yesclosedate_1 = BSI_CreditReporting_yesclosedate.merge(
        yes_reportclosedate[['ACCT_NBR', 'start_date', 'end_date']],
        how='left',
        left_on='LoanNumber',
        right_on='ACCT_NBR'
    )

    # Update 'CREDIT_BUREAU_PAYMENT_HISTORY' values
    valid_values = ['B', 'D', '.', 0, 1, 2, 3, 4, 5, 6, 'H']
    BSI_CreditReporting_yesclosedate_1['CREDIT_BUREAU_PAYMENT_HISTORY'] = np.where(
        BSI_CreditReporting_yesclosedate_1['CREDIT_BUREAU_PAYMENT_HISTORY'].isin(valid_values),
        BSI_CreditReporting_yesclosedate_1['CREDIT_BUREAU_PAYMENT_HISTORY'],
        0
    )
    
    # Convert 'Year_Month_Obs' to datetime format
    BSI_CreditReporting_yesclosedate_1['ymo'] = BSI_CreditReporting_yesclosedate_1['Year_Month_Obs'].apply(
        lambda x: dt.strptime(x, '%Y-%m')
    )

    # Filter rows based on 'ymo' within the 'start_date' and 'end_date' range
    BSI_CreditReporting_yesclosedate_1_ = BSI_CreditReporting_yesclosedate_1[
        (BSI_CreditReporting_yesclosedate_1['ymo'] >= BSI_CreditReporting_yesclosedate_1['start_date']) &
        (BSI_CreditReporting_yesclosedate_1['ymo'] <= BSI_CreditReporting_yesclosedate_1['end_date'])
    ]

    # Create 'CBH' column: 0 for specific categories, 1 otherwise
    BSI_CreditReporting_yesclosedate_1_['CBH'] = np.where(
        BSI_CreditReporting_yesclosedate_1_['CREDIT_BUREAU_PAYMENT_HISTORY'].isin(['B', 'D', '.', 0]),
        0,
        1
    )

    # Summarize CBH by LoanNumber
    BSI_CreditReporting_yesclosedate_final = BSI_CreditReporting_yesclosedate_1_[['LoanNumber', 'CBH']]
    BSI_CreditReporting_yesclosedate_final = BSI_CreditReporting_yesclosedate_final.groupby('LoanNumber').sum('CBH').reset_index()

    return BSI_CreditReporting_yesclosedate_final

# Example usage
# Assuming transform_credit_report_to_time_series is defined elsewhere
file_path = root_directory+'BSI_CREDITREPORTING.xlsx'
result = process_credit_report(file_path, transform_credit_report_to_time_series)
print(result.head())



After implementing CB_RPTD_STA, CB_RPTG_STA and CLOSE_CD_DB000 check
(174776, 56)
>>>>> Credit Report Transformed to Timeseries <<<<<<
   LoanNumber  CBH
0         791    0
1        1235    3
2        1262    0
3        1571    0
4        1596    0


In [ ]:
def rename_and_transform_logins_columns(logins):
    """
    Rename and transform columns in a DataFrame representing login data.

    Args:
        logins (pd.DataFrame): A DataFrame containing login data with columns 'LoanNumber' and 'Date'.

    Returns:
        pd.DataFrame: A DataFrame with columns renamed and additional columns added:
            - 'LoanNumberPlain': Renamed from 'LoanNumber'.
            - 'LoginDate': Converted to a datetime format from the original 'Date'.
            - 'Login_Year_Month': New column derived from 'LoginDate' with the format 'YYYY-MM'.

    Note:
        This function modifies the input DataFrame in place.
    """
    logins.rename(columns={'LoanNumber': 'LoanNumberPlain',
                           'Date': 'LoginDate'}, inplace=True)
    
    logins['LoginDate']=pd.to_datetime(logins['LoginDate'],format = 'mixed')
    logins['Login_Year_Month']=logins['LoginDate'].dt.strftime('%Y-%m')
    logins['LoanNumberPlain']=logins['LoanNumberPlain'].astype(int)
    print('>>>>> Logins columns renamed and transformed <<<<<<')
    return logins


def count_monthly_logins(Logins):
    """
    Count the number of logins per month for each LoanNumberPlain.

    This function uses the variable 'Logins' to generate a DataFrame
    containing the counts of logins for each LoanNumberPlain in each Login_Year_Month.

    Returns:
    DataFrame: A DataFrame containing the columns 'LoanNumberPlain',
               'Login_Year_Month', and 'Login_Count' representing the monthly
               login counts.

    Note:
    - The 'Logins' variable is assumed to be available for SQL queries.
    - The generated DataFrame includes counts of logins grouped by
      LoanNumberPlain and Login_Year_Month.
    - The function also prints a message indicating that the monthly login count
      has been generated.
    """
    Monthly_Login_Count=sqldf.run('''
                                SELECT LoanNumberPlain, Login_Year_Month, count(Login_Year_Month) as Login_Count
                                FROM Logins
                                GROUP BY LoanNumberPlain,  Login_Year_Month
                                '''
                                )
    print('>>>>> Monthly Login Count Generated <<<<<<')
    return Monthly_Login_Count

logins_df = pd.read_csv(root_directory+'MLW_logins.csv')
Logins = rename_and_transform_logins_columns(logins_df)
display(Logins.head())
Monthly_Login_Count = count_monthly_logins(Logins)
display(Monthly_Login_Count)
Monthly_Login_Count['LoanNumberPlain']=Monthly_Login_Count['LoanNumberPlain'].astype(str).str.replace(r'^-','',regex=True)


>>>>> Logins columns renamed and transformed <<<<<<


,LoanNumberPlain,LoginDate,Login_Year_Month
0,1119,2021-02-26 13:22:54.343,2021-02
1,1119,2021-02-28 14:17:27.033,2021-02
2,1119,2021-11-30 10:54:51.877,2021-11
3,1119,2022-02-22 18:14:32.110,2022-02
4,1119,2022-06-26 13:35:29.967,2022-06


>>>>> Monthly Login Count Generated <<<<<<


In [ ]:
print(McK_Oneoff.shape)
print(BSI_CreditReporting_yesclosedate_final.shape)
McK_Oneoff_CBH = McK_Oneoff.merge(BSI_CreditReporting_yesclosedate_final,how='left',on = 'LoanNumber')
print(McK_Oneoff_CBH.shape)
McK_Oneoff_CBH.head()
Training_Oneoff=McK_Oneoff_CBH.merge(Avg_interest_rate, how='left',left_on=['LoanNumber'],right_on=['LoanNumber'])
Training_Oneoff_Data = Training_Oneoff.merge(Monthly_Login_Count, how='left',left_on=['LoanNumber','Year_Month_Obs'],right_on=['LoanNumberPlain','Login_Year_Month'])
display(Training_Oneoff_Data.head())

In [ ]:
Training_Oneoff_Data_Record = add_fips_to_training_oneoff(Training_Oneoff_Data, public_dataframes['Zip_fips_lookup'])
display(Training_Oneoff_Data_Record.head())
Demographic = clean_domographic(public_dataframes['Demographic'])
display(Demographic.head())

Training_Oneoff_Data_Record=Training_Oneoff_Data_Record.merge(Demographic, how='left',left_on=['FIPS'],right_on=['geography'])
Training_Oneoff_Data_Record=Training_Oneoff_Data_Record.merge(Purchase_index, how='left',left_on=['Year_Month_Obs','PropertyState'],right_on=['Year_Month_Index','state'])
display(Training_Oneoff_Data_Record.head())
print(Training_Oneoff_Data_Record.shape)

Training_Oneoff_Data_Record.rename(columns = {'CBH':'CREDIT_BUREAU_PAYMENT_HISTORY'}, inplace = True)


In [ ]:
# Transformation a New Features for used the Model Training

Training_Oneoff_Data_Record['Is_MailingZip_Equal_PropZip']=(Training_Oneoff_Data_Record['PropertyZipCode'].
                                                            astype(str)==Training_Oneoff_Data_Record['MailingZip'].
                                                            astype(str).apply(lambda x: x[:6])).astype(int)

Training_Oneoff_Data_Record['Interest_Delta_Current_Orginal']=(Training_Oneoff_Data_Record['InterestRate']-
                                                               Training_Oneoff_Data_Record['OriginalInterestRate']) # This features not used in model training, just for analysis to check.

Training_Oneoff_Data_Record['Interest_Delta_Current_Market']=(Training_Oneoff_Data_Record['InterestRate']-
                                             Training_Oneoff_Data_Record['Orginal_Interest_Rate'])

Training_Oneoff_Data_Record['Ltv_Delta_Orginal_Current']=(Training_Oneoff_Data_Record['Original_Ltv_Calc']-
                                         Training_Oneoff_Data_Record['CurrentLTV'])

Training_Oneoff_Data_Record['CREDIT_BUREAU_PAYMENT_HISTORY']=Training_Oneoff_Data_Record['CREDIT_BUREAU_PAYMENT_HISTORY'].fillna(0)

In [ ]:
# Select the model training feature
Training_Oneoff_Data_Record=Training_Oneoff_Data_Record[["LoanNumber","LoanType","CREDIT_BUREAU_PAYMENT_HISTORY",
                                                        "PrincipalBalance","Is_MailingZip_Equal_PropZip","FICO_Score",
                                                        "CurrentLTV","Login_Count","Interest_Delta_Current_Market",
                                                         "Ltv_Delta_Orginal_Current",
                                                         "Percent_EDUCATIONAL_ATTAINMENT_Population_25_years_and_over_Bachelors_degree_or_higher",
                                                        "Estimate_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Total_households_Mean_household_income_dollars",
                                                        "Percent_HOUSING_TENURE_Occupied_housing_units_Owner_occupied"]]

In [ ]:
# a function that reads all .csv and .xlsx files in Data directory and returns a dictionary of DataFrames 
# with filenames as keys and loaded datafrmaes as values

def read_files_in_data_directory(directory_path, is_bsi_data=True):
    # Ensure the directory path is valid
    if not os.path.isdir(directory_path):
        raise ValueError("Invalid directory path")

    # Initialize an empty dictionary to store DataFrames
    dataframes = {}
    specific_month = ''
    
    # Loop through files in the directory
    for filename in tqdm(os.listdir(directory_path), desc ="Loading Data Progress"):
        file_path = os.path.join(directory_path, filename)

        # Check if the file is a CSV or Excel file
        if filename.lower().endswith(('.csv', '.xlsx')):
            # Read the file into a DataFrame
            if filename.lower().endswith('.csv'):
                df = pd.read_csv(file_path, low_memory=False) # , error_bad_lines = False
            elif filename.lower().endswith('.xlsx'):
                df = pd.read_excel(file_path)
            
            # if 'oneoff' in filename.lower():
            if any(key_word in filename.lower() for key_word in ['one', 'off']):
                raw_data_info = filename[:-4]
                dataframe_name = 'McK_Oneoff'
                pattern = '|'.join(map(re.escape, ['_', '.', '-']))
                # Use re.split to split the string based on the delimiters
                result = re.split(pattern, filename)
                specific_month = '_'.join(result[-3:-1]).upper() + '_'
            elif 'credit' in filename.lower():
                dataframe_name = 'BSI_CREDITREPORTING'
            elif 'mlw' in filename.lower():
                dataframe_name = 'Logins'
            
            elif 'interest' in filename.lower():
                dataframe_name = 'Avg_interest_rate'
            elif 'demographic' in filename.lower():
                dataframe_name = 'Demographic'
            elif 'zip' in filename.lower():
                dataframe_name = 'Zip_fips_lookup'
            elif 'hpi' or 'house' in filename.lower():
                dataframe_name = 'Purchase_index'
            else:
                dataframe_name = os.path.splitext(filename)[0]

            print(f"{dataframe_name} is loaded: {df.shape}")
            print()
            # Add the DataFrame to the dictionary with the filename as the key
            dataframes[dataframe_name] = df
            
    if is_bsi_data:
        bsi_or_public = 'BSI'
    else:
        bsi_or_public = 'Public'
    print(f">>>>>> Loading {bsi_or_public} Data Completed <<<<<<<")
    return dataframes, specific_month, raw_data_info if is_bsi_data else dataframes

def check_dataframes(BSI_dataframes, public_dataframes, needed_dataframes = ['McK_Oneoff',
                                                                            'BSI_CREDITREPORTING',
                                                                            'Logins',
                                                                            'Avg_interest_rate',
                                                                            'Zip_fips_lookup',
                                                                            'Purchase_index',
                                                                            'Demographic']):
    
    combined_dataframe_dictionaries = {**BSI_dataframes,**public_dataframes}
    #  Check if all the needed dataframes exist
    try:
        for df_name in needed_dataframes:
            if df_name not in combined_dataframe_dictionaries.keys():
                raise NameError(f"{df_name} data not found!")
        print(">>>>>> All needed dataframes found <<<<<<")
    except NameError as e:
        print(f"Error: {e}")
        raise  # Raise the exception to halt the program


def add_year_month_and_filter_oneoff(McK_Oneoff, cols=None):
    """
    

    Parameters:
    - McK_Oneoff (pd.DataFrame): The input DataFrame containing loan data.
    - cols (list, optional): A list of column names to be selected in the final DataFrame.
                             If not provided, default columns will be selected.

    Returns:
    - pd.DataFrame: A DataFrame containing the latest occurrence of each loan with selected columns.

    Steps:
    1. Converts the 'DataDate' column to datetime format.
    2. Identifies the index of the last occurrence for each 'LoanNumber' based on 'DataDate'.
    3. Filters the DataFrame to include only the rows with the identified indices.
    4. Adds a new column 'Year_Month_Obs' formatted as '%Y-%m'.
    5. Resets the DataFrame index.
    6. Selects and retains specific columns from the DataFrame.

    Note:
    The function modifies the input DataFrame in-place and prints a message indicating the operation.
    """

    McK_Oneoff['DataDate']=pd.to_datetime(McK_Oneoff['DataDate'],format='mixed')
    # last_occurrence_index = McK_Oneoff.groupby('LoanNumber')['DataDate'].idxmax()
    # McK_Oneoff = McK_Oneoff[McK_Oneoff.index.isin(last_occurrence_index)].reset_index(drop=True)
    # clean and format
    McK_Oneoff['Year_Month_Obs']=McK_Oneoff['DataDate'].dt.strftime('%Y-%m')
    McK_Oneoff=McK_Oneoff.reset_index(drop=True)

    print(">>>>> Year_Month_Obs column added to McK_Oneoff  <<<<<")

    # select columns
    if not cols:
        cols=[
            'LoanNumber',
            'LoanType',
            'PrincipalBalance',
            'NextDueDate',
            'InterestPaidToDate',
            'DateLastPayment',
            'InterestRate',
            'OriginalInterestRate',
            'PropertyCityName',
            'PropertyState',
            'PropertyZipCode',
            'MailingCity',
            'MailingState',
            'MailingZip',
            'FICO_Score',
            'FicoDate',
            'BankruptcyFlag',
            'LossMitigationStatus',
            'ForeclosureStatus',
            'DLQ_Status',
            'NoteDate',
            'ServicingDate',
            'TransferDate',
            'PiPmt',
            'EscrowPayment',
            'EscrowBalance',
            'ReportToCreditBureau',
            'AppraisalDate',
            'AppraisedAmount',
            'Original_Ltv_Calc',
            'CurrentLTV',
            'PriorServicerName',
            'InvestorOwner',
            'AcquiredPortfolioName',
            'PrimaryBorrowerOptOut',
            'SecondaryBorrowerOptOut',
            'TermOfLoan',
            'RemainingTerm',
            'FIXED ARM',
            'SaleAmount',
            'OriginalBalance',
            'RawLeadType',
            'ForbearanceFlag',
            'Year_Month_Obs'
        ]

    McK_Oneoff = McK_Oneoff[cols]
    print(">>>>> McK_Oneoff filtered for the needed columns <<<<<")

    return  McK_Oneoff 

def transform_credit_report_to_time_series(BSI_CreditReporting, Year_Month_Obs=None):
    print("After implementing CB_RPTD_STA, CB_RPTG_STA and CLOSE_CD_DB000 check")
    filtered_df = BSI_CreditReporting.copy()
    print(filtered_df.shape)
    if not Year_Month_Obs:
        Year_Month_Obs=[
            '2023-01',
            '2023-02',
            '2023-03',
            '2023-04',
            '2023-05',
            '2023-06',
            '2023-07',
            '2023-08',
            '2023-09',
            '2023-10',
            '2023-11',
            '2023-12',
            '2022-01',
            '2022-02',
            '2022-03',
            '2022-04',
            '2022-05',
            '2022-06',
            '2022-07',
            '2022-08',
            '2022-09',
            '2022-10',
            '2022-11',
            '2022-12',
            '2021-01',
            '2021-02',
            '2021-03',
            '2021-04',
            '2021-05',
            '2021-06',
            '2021-07',
            '2021-08',
            '2021-09',
            '2021-10',
            '2021-11',
            '2021-12',
        ]
 
    All_LoanNumber=[]
    All_Year_Month_Obs=[]
    All_History=[]
    for loan in filtered_df['ACCT_NBR']:
        History=filtered_df[filtered_df['ACCT_NBR']==loan].iloc[:,-37:-1]
        LoanNumber=[loan]*36
        x1=LoanNumber
        x2=Year_Month_Obs
        x3=History.iloc[0,:].to_list()
        All_LoanNumber.extend(x1)
        All_Year_Month_Obs.extend(x2)
        All_History.extend(x3)
 
    data = {'LoanNumber': All_LoanNumber,
            'Year_Month_Obs': All_Year_Month_Obs,
            'CREDIT_BUREAU_PAYMENT_HISTORY':All_History}
    BSI_CreditReporting = pd.DataFrame()
    BSI_CreditReporting = pd.DataFrame(data)
    print('>>>>> Credit Report Transformed to Timeseries <<<<<<')
    return BSI_CreditReporting
 

def rename_and_transform_logins_columns(logins):
    """
    Rename and transform columns in a DataFrame representing login data.

    Args:
        logins (pd.DataFrame): A DataFrame containing login data with columns 'LoanNumber' and 'Date'.

    Returns:
        pd.DataFrame: A DataFrame with columns renamed and additional columns added:
            - 'LoanNumberPlain': Renamed from 'LoanNumber'.
            - 'LoginDate': Converted to a datetime format from the original 'Date'.
            - 'Login_Year_Month': New column derived from 'LoginDate' with the format 'YYYY-MM'.

    Note:
        This function modifies the input DataFrame in place.
    """
    logins.rename(columns={'LoanNumber': 'LoanNumberPlain',
                           'Date': 'LoginDate'}, inplace=True)
    
    logins['LoginDate']=pd.to_datetime(logins['LoginDate'],format = 'mixed')
    logins['Login_Year_Month']=logins['LoginDate'].dt.strftime('%Y-%m')
    logins['LoanNumberPlain']=logins['LoanNumberPlain'].astype(int)
    print('>>>>> Logins columns renamed and transformed <<<<<<')
    return logins


def count_monthly_logins(Logins):
    """
    Count the number of logins per month for each LoanNumberPlain.

    This function uses the variable 'Logins' to generate a DataFrame
    containing the counts of logins for each LoanNumberPlain in each Login_Year_Month.

    Returns:
    DataFrame: A DataFrame containing the columns 'LoanNumberPlain',
               'Login_Year_Month', and 'Login_Count' representing the monthly
               login counts.

    Note:
    - The 'Logins' variable is assumed to be available for SQL queries.
    - The generated DataFrame includes counts of logins grouped by
      LoanNumberPlain and Login_Year_Month.
    - The function also prints a message indicating that the monthly login count
      has been generated.
    """
    Monthly_Login_Count=sqldf.run('''
                                SELECT LoanNumberPlain, Login_Year_Month, count(Login_Year_Month) as Login_Count
                                FROM Logins
                                GROUP BY LoanNumberPlain,  Login_Year_Month
                                '''
                                )
    print('>>>>> Monthly Login Count Generated <<<<<<')
    return Monthly_Login_Count

def add_fips_to_training_oneoff(training_oneoff, Zip_fips_lookup): # Take two argument Mck_Oneoff & add_fips
    training_oneoff['FIPS'] = np.nan # Mck_Oneoff dataset create a column name FIPS fills NaN.
    for i in range(training_oneoff.shape[0]): # Iterate the mck_oneoff data from row 0 to till end. use of shape[0].
    # for i in range(30):
        zip = int(training_oneoff.loc[i,'PropertyZipCode'][-5:]) # Taking the last character propertyzip column of the current row in mck_oneoff, convert int
        row = Zip_fips_lookup.index[Zip_fips_lookup['ZIP'] == zip] # Find the index(row number) in the Zips_fips dataframe where the ZIP column matches the extracted zip code.
        if len(row)==0: # Check the zip code ==0 then pass the else condition, if true then store the fips code in fips column
            pass
        else:
            row = row[0]
            fips = Zip_fips_lookup.loc[row,'STCOUNTYFP'] 
            training_oneoff.loc[i,'FIPS'] = fips
    print('>>>>> FIPS Added to McK_Oneoff <<<<<<')
    return training_oneoff


def clean_domographic(Demographic, keep_only_needed_column=False):
    Demographic = Demographic.loc[:, ~Demographic.columns.str.contains('^Unnamed')]
    columns=[
    'code',
    'geography',
    'Percent_EDUCATIONAL_ATTAINMENT_Population_25_years_and_over_Bachelors_degree_or_higher',
    'Percent_VEHICLES_AVAILABLE_Occupied_housing_units_No_vehicles_available',
    'Percent_VEHICLES_AVAILABLE_Occupied_housing_units_1_vehicle_available',
    'Percent_VEHICLES_AVAILABLE_Occupied_housing_units_3_or_more_vehicles_available',
    'Percent_INDUSTRY_Civilian_employed_population_16_years_and_over_Agriculture_forestry_fishing_and_hunting_and_mining',
    'Percent_INDUSTRY_Civilian_employed_population_16_years_and_over_Construction',
    'Percent_INDUSTRY_Civilian_employed_population_16_years_and_over_Manufacturing',
    'Percent_INDUSTRY_Civilian_employed_population_16_years_and_over_Wholesale_trade',
    'Percent_INDUSTRY_Civilian_employed_population_16_years_and_over_Retail_trade',
    'Percent_INDUSTRY_Civilian_employed_population_16_years_and_over_Transportation_and_warehousing_and_utilities',
    'Percent_INDUSTRY_Civilian_employed_population_16_years_and_over_Information',
    'Percent_INDUSTRY_Civilian_employed_population_16_years_and_over_Finance_and_insurance_and_real_estate_and_rental_and_leasing',
    'Percent_INDUSTRY_Civilian_employed_population_16_years_and_over_Professional_scientific_and_management_and_administrative_and_waste_management_services',
    'Percent_INDUSTRY_Civilian_employed_population_16_years_and_over_Educational_services_and_health_care_and_social_assistance',
    'Percent_INDUSTRY_Civilian_employed_population_16_years_and_over_Arts_entertainment_and_recreation_and_accommodation_and_food_services',
    'Percent_INDUSTRY_Civilian_employed_population_16_years_and_over_Other_services_except_public_administration',
    'Percent_INDUSTRY_Civilian_employed_population_16_years_and_over_Public_administration',
    'Percent_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Total_households_With_earnings',
    'Percent_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Total_households_With_Social_Security',
    'Percent_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Total_households_With_retirement_income',
    'Percent_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Total_households_With_Supplemental_Security_Income',
    'Percent_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Total_households_With_cash_public_assistance_income',
    'Percent_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Total_households_With_Food_Stamp_over_SNAP_benefits_in_the_past_12_months',
    'Percent_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Nonfamily_households',
    'Estimate_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Total_households_Median_household_income_dollars',
    'Estimate_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Total_households_Mean_household_income_dollars',
    'Estimate_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Total_households_With_earnings_Mean_earnings_dollars',
    'Estimate_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Total_households_With_Social_Security_Mean_Social_Security_income_dollars',
    'Estimate_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Total_households_With_retirement_income_Mean_retirement_income_dollars',
    'Estimate_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Total_households_With_Supplemental_Security_Income_Mean_Supplemental_Security_Income_dollars',
    'Estimate_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Total_households_With_cash_public_assistance_income_Mean_cash_public_assistance_income_dollars',
    'Percent_HEALTH_INSURANCE_COVERAGE_Civilian_noninstitutionalized_population_With_health_insurance_coverage_With_private_health_insurance',
    'Percent_HEALTH_INSURANCE_COVERAGE_Civilian_noninstitutionalized_population_With_health_insurance_coverage_With_public_coverage',
    'Percent_HEALTH_INSURANCE_COVERAGE_Civilian_noninstitutionalized_population_No_health_insurance_coverage',
    'Estimate_EMPLOYMENT_STATUS_Population_16_years_and_over',
    'Percent_EMPLOYMENT_STATUS_Population_16_years_and_over_In_labor_force',
    'Percent_EMPLOYMENT_STATUS_Civilian_labor_force_Unemployment_Rate',
    'Percent_SELECTED_MONTHLY_OWNER_COSTS_AS_A_PERCENTAGE_OF_HOUSEHOLD_INCOME_SMOCAPI_Housing_units_with_a_mortgage_excluding_units_where_SMOCAPI_cannot_be_computed_35_0_percent_or_more',
    'Percent_SELECTED_MONTHLY_OWNER_COSTS_AS_A_PERCENTAGE_OF_HOUSEHOLD_INCOME_SMOCAPI_Housing_unit_without_a_mortgage_excluding_units_where_SMOCAPI_cannot_be_computed_35_0_percent_or_more',
    'Percent_GROSS_RENT_AS_A_PERCENTAGE_OF_HOUSEHOLD_INCOME_GRAPI_Occupied_units_paying_rent_excluding_units_where_GRAPI_cannot_be_computed_35_0_percent_or_more',
    'Percent_HOUSING_TENURE_Occupied_housing_units_Owner_occupied',
    'Percent_HOUSING_TENURE_Occupied_housing_units_Renter_occupied',
    'Percent_HOUSING_OCCUPANCY_Total_housing_units_Vacant_housing_units',
    'GINI_index',
    'Estimate_Median_gross_rent_as_a_percentage_of_household_income',
    'Estimate_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Total_households_Median_household_income_dollars_1',
    'Estimate_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Total_households_Mean_household_income_dollars_1',
    'Estimate_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Total_households_With_earnings_Mean_earnings_dollars_1',
    'Estimate_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Total_households_With_Social_Security_Mean_Social_Security_income_dollars_1',
    'Estimate_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Total_households_With_retirement_income_Mean_retirement_income_dollars_1',
    'Estimate_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Total_households_With_Supplemental_Security_Income_Mean_Supplemental_Security_Income_dollars_1',
    'Estimate_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Total_households_With_cash_public_assistance_income_Mean_cash_public_assistance_income_dollars_1',
    'Estimate_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Families_Median_family_income_dollars',
    'Estimate_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Families_Mean_family_income_dollars',
    'Estimate_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Per_capita_income_dollars',
    'Estimate_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Nonfamily_households_Median_nonfamily_income_dollars',
    'Estimate_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Nonfamily_households_Mean_nonfamily_income_dollars',
    'Estimate_INCOME_AND_BENEFITS_IN_2021_INFLATION_ADJUSTED_DOLLARS_Median_earnings_for_workers_dollars'
    ]
    Demographic.columns=columns

    
    return Demographic




In [ ]:
# Paths to directories
BSI_data_path = os.path.join(root_directory, 'BSI_data/')
public_data_path = os.path.join(root_directory, 'Public_data/')


BSI_dataframes, specific_month, raw_data_info = read_files_in_data_directory(BSI_data_path) # read all BSI dataframes
 
globals().update(BSI_dataframes) # this line creates variables from dataframes keys with the same name
print('Month of Observation:', specific_month)
BSI_dataframe_names = list(BSI_dataframes.keys())
# del BSI_dataframes
print('Extracted specific month:', specific_month)
 
public_dataframes, _, _ = read_files_in_data_directory(public_data_path, False) # read all public dataframes
print(public_dataframes.keys())
globals().update(public_dataframes) # this line creates variables from dataframes keys with the same name
public_dataframe_names = list(public_dataframes.keys())
 
check_dataframes(BSI_dataframes, public_dataframes)
 
McK_Oneoff_1yr = add_year_month_and_filter_oneoff(BSI_dataframes['McK_Oneoff'])
 
print(McK_Oneoff_1yr.shape)
display(McK_Oneoff_1yr.head())

In [ ]:
# Model Training 1 yr data from Sep-22 to Aug-23

McK_Oneoff_1yr['Year_Month_Obs'] = pd.to_datetime(McK_Oneoff_1yr['Year_Month_Obs'])
print(McK_Oneoff_1yr.shape)
 
import functools
def disjunction_and(*conditions):
    return functools.reduce(np.logical_and, conditions)
 
c_1 = McK_Oneoff_1yr['Year_Month_Obs']>'2022-08' # Sep-2022 to Start
c_2 =McK_Oneoff_1yr['Year_Month_Obs']<='2023-08' # Aug-2023 to End
 
McK_Oneoff = McK_Oneoff_1yr[disjunction_and(c_1,c_2)].reset_index(drop=True)
print(McK_Oneoff.shape)
print(McK_Oneoff['Year_Month_Obs'].value_counts())

In [ ]:

McK_Oneoff = McK_Oneoff.sort_values(['Year_Month_Obs']).groupby(['LoanNumber']).last().reset_index()
print(McK_Oneoff.shape)
McK_Oneoff.head(2)

In [ ]:
# Filter the data from ['CBR_LAST_REPORT_DATE']>'2022-01']
# Take the data based on CBR_LAST_REPORT_DATE and Lag 2 month

yes_reportclosedate = BSI_CREDITREPORTING[BSI_CREDITREPORTING['CBR_LAST_REPORT_DATE']>'2022-01'].reset_index()
yes_reportclosedate['start_date'] = yes_reportclosedate.CBR_LAST_REPORT_DATE.apply(lambda x: dt.strptime(dt.strftime(x + relativedelta(months = -13),'%Y-%m'), '%Y-%m'))
yes_reportclosedate['end_date'] = yes_reportclosedate.CBR_LAST_REPORT_DATE.apply(lambda x: dt.strptime(dt.strftime(x + relativedelta(months = -2),'%Y-%m'), '%Y-%m'))
yes_reportclosedate.head()

In [ ]:
# Transform vertically all loan number with Year_Month_Obs with merging

BSI_CreditReporting_yesclosedate = transform_credit_report_to_time_series(yes_reportclosedate)
print(BSI_CreditReporting_yesclosedate.head(2))
BSI_CreditReporting_yesclosedate_1 = BSI_CreditReporting_yesclosedate.merge(yes_reportclosedate[['ACCT_NBR','start_date','end_date']],how='left',left_on='LoanNumber',right_on='ACCT_NBR')
print(BSI_CreditReporting_yesclosedate_1.head(2))

In [ ]:
BSI_CreditReporting_yesclosedate_1['CREDIT_BUREAU_PAYMENT_HISTORY'] = np.where(BSI_CreditReporting_yesclosedate_1['CREDIT_BUREAU_PAYMENT_HISTORY'].isin(['B','D','.',0,1,2,3,4,5,6,'H']),BSI_CreditReporting_yesclosedate_1['CREDIT_BUREAU_PAYMENT_HISTORY'],0)
BSI_CreditReporting_yesclosedate_1['ymo'] = BSI_CreditReporting_yesclosedate_1['Year_Month_Obs'].apply(lambda x: dt.strptime(x, '%Y-%m'))
# Filtering data on the basis of start and end date comparing with ymo
BSI_CreditReporting_yesclosedate_1_ = BSI_CreditReporting_yesclosedate_1[BSI_CreditReporting_yesclosedate_1['ymo'] >= BSI_CreditReporting_yesclosedate_1['start_date']]
BSI_CreditReporting_yesclosedate_1_ = BSI_CreditReporting_yesclosedate_1_[BSI_CreditReporting_yesclosedate_1_['ymo'] <=BSI_CreditReporting_yesclosedate_1_['end_date']]
 
# Removing timestamp from CREDIT_BUREAU_PAYMENT_HISTORY feature
BSI_CreditReporting_yesclosedate_1_['CREDIT_BUREAU_PAYMENT_HISTORY'] = np.where(BSI_CreditReporting_yesclosedate_1_['CREDIT_BUREAU_PAYMENT_HISTORY'].isin(['B','D','.',0,1,2,3,4,5,6,'H']),BSI_CreditReporting_yesclosedate_1_['CREDIT_BUREAU_PAYMENT_HISTORY'],0)
 
# Convert 0 if CBH belongs to categories like (B, D, 0, .) else 1
BSI_CreditReporting_yesclosedate_1_['CBH'] = np.where(BSI_CreditReporting_yesclosedate_1_['CREDIT_BUREAU_PAYMENT_HISTORY'].isin(['B','D','.',0]),0,1)
 
print(BSI_CreditReporting_yesclosedate_1_.shape)
BSI_CreditReporting_yesclosedate_1_.head(2)

In [ ]:
BSI_CreditReporting_yesclosedate_final = BSI_CreditReporting_yesclosedate_1_[['LoanNumber','CBH']]
BSI_CreditReporting_yesclosedate_final = BSI_CreditReporting_yesclosedate_final.groupby('LoanNumber').sum('CBH')
#BSI_CreditReporting_yesclosedate_final.rename(columns = {'dummy':'CREDIT_BUREAU_PAYMENT_HISTORY'}, inplace=True)
BSI_CreditReporting_yesclosedate_final = BSI_CreditReporting_yesclosedate_final.reset_index()
BSI_CreditReporting_yesclosedate_final.head()

In [ ]:
print(McK_Oneoff.shape)
print(BSI_CreditReporting_yesclosedate_final.shape)
McK_Oneoff_CBH = McK_Oneoff.merge(BSI_CreditReporting_yesclosedate_final,how='left',on = 'LoanNumber')
print(McK_Oneoff_CBH.shape)
McK_Oneoff_CBH.head()

In [ ]:
Training_Oneoff=McK_Oneoff_CBH.merge(Avg_interest_rate.drop(columns=['LoanType']), how='left',left_on=['LoanNumber'],right_on=['LoanNumber'])

In [ ]:
Logins = rename_and_transform_logins_columns(BSI_dataframes['Logins'])
display(Logins.head())
Monthly_Login_Count = count_monthly_logins(Logins)
display(Monthly_Login_Count)
Monthly_Login_Count['LoanNumberPlain']=Monthly_Login_Count['LoanNumberPlain'].astype(str).str.replace(r'^-','',regex=True)
Monthly_Login_Count['LoanNumberPlain']=Monthly_Login_Count["LoanNumberPlain"].astype(int)

In [ ]:
Training_Oneoff_Data = Training_Oneoff.merge(Monthly_Login_Count, how='left',left_on=['LoanNumber','Year_Month_Obs'],right_on=['LoanNumberPlain','Login_Year_Month'])
display(Training_Oneoff_Data.head())

In [ ]:
Training_Oneoff_Data_Record = add_fips_to_training_oneoff(Training_Oneoff_Data, public_dataframes['Zip_fips_lookup'])
display(Training_Oneoff_Data_Record.head())
Demographic = clean_domographic(public_dataframes['Demographic'])
display(Demographic.head())
Training_Oneoff_Data_Record=Training_Oneoff_Data_Record.merge(Demographic, how='left',left_on=['FIPS'],right_on=['geography'])
Training_Oneoff_Data_Record=Training_Oneoff_Data_Record.merge(Purchase_index, how='left',left_on=['Year_Month_Obs','PropertyState'],right_on=['Year_Month_Index','state'])
display(Training_Oneoff_Data_Record.head())
print(Training_Oneoff_Data_Record.shape)